# Out-of-Equilibrium Structure Identification in LeMat-Bulk Dataset

This notebook identifies and analyzes out-of-equilibrium structures in the LeMat-Bulk dataset. It loads the dataset, computes the maximum atomic force for each structure, and classifies structures based on force magnitude thresholds. The indices of structures exceeding each threshold are saved for further study or filtering, enabling targeted analysis of non-equilibrium configurations.

## Set up imports and development environment

This section imports the necessary libraries for structural analysis and sets up the development environment. We import the BAWL hasher for structural fingerprinting and the Pymatgen structure similarity module for comparing crystal structures.

In [ ]:
# Set up imports and autoreload for development.
from material_hasher.hasher.bawl import BAWLHasher
from material_hasher.similarity import PymatgenStructureSimilarity

%load_ext autoreload
%autoreload 2

## Load the LeMat-Bulk dataset from Hugging Face Datasets

We load the training split for the LeMat-Bulk dataset. It includes structural information such as lattice vectors, atomic positions, and chemical compositions.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("LeMaterial/LeMat-Bulk", "compatible_pbe", split="train")

## Display Force Threshold Statistics

This cell displays the count of structures exceed the force magnitude thresholds of 0.2 eV/Å and shows how many structuresare included. This information helps understand the distribution of out-of-equilibrium structures in the dataset and can guide the selection of appropriate force thresholds for filtering or analysis purposes.

In [ ]:
import numpy as np
import csv

high_force_indices = []
for idx in range(len(dataset)):
    forces = dataset[idx]["forces"]
    forces_array = np.array(forces)

    # Skip if it's not 2D with shape (N_atoms, 3)
    if forces_array.ndim != 2 or forces_array.shape[1] != 3:
        continue

    max_norm_force = np.max(np.linalg.norm(forces_array, axis=1))

    if max_norm_force > 0.2:
        high_force_indices.append([idx])
        print(idx, max_norm_force)

with open("zero_force_indices.csv", mode="w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["index"])  # header
    writer.writerows(high_force_indices)